In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

# We set-up a simple time-independent two-well system

We start with the usual equation of the resonant flux-biased ATS Hamiltonian in the RWA
\begin{equation}
    \begin{cases}
    \dot a = -i\beta_a'a^*b\\
    \dot b = -i\beta_b'a^2/2-i\varepsilon_d'/2-\kappa_b'b
    \end{cases}
\end{equation}
Then, we perform adiabatic elimination $\dot b = 0$ and we get $$\dot a = -i\beta_a'a^*(i\beta_b'a^2/2\kappa_b'-i\varepsilon_d'/2\kappa_b')$$

We introduce the quadratures $x=(a+a^*)/\sqrt 2$ and $p=i(a^*-a)/\sqrt 2$ 
\begin{equation}
    \begin{cases}
    \dot x = - \aleph (x^2+p^2)x - \beth x\\
    \dot p = - \aleph (x^2+p^2)p + \beth p
    \end{cases},
\end{equation}
with $\aleph = \beta_a'\beta_b'/4\kappa_b'$ and $\beth = \beta_a'\varepsilon_d'/2\kappa_b'$.

Then we can define a potential $U(x,p)$ so that $\dot x = \partial_xU$ and $\dot p = \partial_pU$ :$$U(x,p) = \aleph(x^2+p^2)^2/4+\beth(x^2-p^2)/2$$



In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2
#### Caracterization of the circuit #####
ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_a^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2pi*1.5

#Environment
κ_b=1/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.2
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 10.12*g2#0.75*g2_max

β_a_p, β_b_p = 2*E_J*sin(ε_p)*φ_a^2/(ω_a), 2*E_J*sin(ε_p)*φ_b^2/(ω_b)
ε_d_p = ε_d*φ_b/ω_b
κ_b_p = κ_b / ω_b

ℵ = β_a_p*β_b_p/(4*κ_b_p)
ℶ = β_a_p*ε_d_p/(2*κ_b_p)

In [ ]:
sqrt(2*ε_d_p/β_b_p)

In [ ]:
function equation!(du,u,params,t)
    ℵ,ℶ = params

    x = u[1]
    p = u[2]
        
    du[1] = -ℵ*(x^2+p^2)*x-ℶ*x
    du[2] = -ℵ*(x^2+p^2)*p+ℶ*p
    return nothing
end

In [ ]:
t_min = 0.0
t_max = 2/κ_b_p 
tspan = (t_min, t_max)
x_0, p_0 = 0.0, -0.001
u_0 = [x_0, p_0]
prob = ODEProblem(equation!, u_0, tspan, (ℵ,ℶ))
sol = solve(prob, Tsit5())
t = sol.t
x = [u[1] for u in sol.u]
p = [u[2] for u in sol.u]
p_stable=p;

In [ ]:
x_all = -.2:0.01:.2
p_all = -.35:0.01:.35

U = [ℵ*(x^2 + p^2)^2/4 + ℶ*(x^2 - p^2)/2 for p in p_all, x in x_all]

heatmap(x_all, p_all, U)
scatter!(x,p, label="Trajectory")

Since the system is only 1D ($x=0$ on the minimizing path and on the dissipative path), we will focus only on $p$. Therfore the equation of motion reads
$$\dot p = -\aleph p^3+\beth p$$

# Writing the Hamiltonian and initializing the GMAM

We now add noise to the system
\begin{equation}
    \dot p = - \aleph p^3 + \beth p + \sqrt \epsilon\eta(t).
\end{equation}

We can write the associated action: $$S=\int dt [P_p\dot p - H(p,P_p)],$$
with the Hamiltonian $$H(p,P_p) = [ - \aleph p^2+\beth]pP_p + \epsilon P_p^2$$

We parametrize time to solve a geometric problem $$\min_{\lambda\geq0}\min_p\max_P\int_0^1[p'P-\lambda^{-1}H(p,P)]ds,$$
and we add a constraint on the parametrization of $p$ : $|dp/ds|=|p'|=cst$

Inputs: $M = 100 \in \mathbb N$, two stable fixed points $\phi_a = (-\sqrt{2\varepsilon_d'/\beta_b'})$ and $\phi_b = (0)$ of the noiseless dynamics where $H(\phi_{a,b},0)=\nabla_{\bm \theta}H(\phi_{a,b},0)=0$; a path $\{\hat\phi_i^0\}_{i\in I}$ with $\hat\phi_0^0 =\phi_a$ and $\hat\phi_M^0 =\phi_b$, such that $|\hat\phi_{i+1}^0-\hat\phi_{i}^0|$ is constant in $i$; the functions $f(\mathbf u, \mathbf v)= \alpha \nabla_{\bm \phi}H - \nabla_{\bm \theta}H$, $g(\mathbf u, \mathbf v)=\alpha \nabla_{\bm \phi}H + \nabla_{\bm \theta}H$ and $\lambda(\mathbf u, \mathbf v) = \biggl[|\langle \nabla_{\bm \theta}H, \partial_s \hat\phi \rangle|+\sqrt{\max(0,\langle \nabla_{\bm \theta}H, \partial_s \hat\phi \rangle^2-4H|\partial_s\hat\phi|^2)}\biggr]/(2|\partial_s\hat\phi|^2)$; $\Delta\tau = 0.1 > 0, \alpha = 1 >0$

We have $\nabla_{\bm\phi}H = \begin{pmatrix}- 3\aleph p^2P_p +\beth P_p \end{pmatrix}$ and $\nabla_{\bm\theta}H = \begin{pmatrix} -\aleph p^3 +\beth p + 2 \epsilon P_p\end{pmatrix}$

# Let's perform the gradient descent ascent

We got that for $n\geq0$, we solve for each $i=M,M-1,\dots,1,0$ for the value of $u_i^{n+1}$ through $$u_M^{n+1} = -v_M^n+2\phi_b,$$ $$ u_i^{n+1} = \biggl[\lambda_i(u^n,v^n)u_{i+1}^{n+1}/\Delta s + f(u_i^n,v_i^n) + u_i^n/\Delta \tau\biggr]/\biggl[1/\Delta\tau + \lambda_i(u^n,v^n)/\Delta s\biggr] , \qquad i=M-1,\dots,1,0$$

For the value of $v_i^{n+1}$ through $$v_0^{n+1} = -u_0^{n+1}+2\phi_a,$$ $$ v_{i}^{n+1} = \biggl[\lambda_i(u^{n+1},v^n)v_{i-1}^{n+1}/\Delta s + g(u_i^{n+1},v_i^n)+v_i^n/\Delta\tau\biggr]/\biggl[1/\Delta\tau + \lambda_i(u^{n+1},v^n)/\Delta s\biggr] , \qquad i=1,\dots,M-1,M$$

In [ ]:
M = 300                    # start smaller, then increase
ϵ = 1e-3
ϕ_a = -sqrt(2*ε_d_p/β_b_p)
ϕ_b = 0.0
ϕ_grid = collect(range(ϕ_a, ϕ_b, length=M+1))

α = 0.01
θ_best = @. (ℵ*ϕ_grid^3 - ℶ*ϕ_grid) / ϵ   # = -b(ϕ)/ϵ
θ_init = zeros(M+1)
θ_init[1] = 0.0
θ_init[end] = 0.0

u = ϕ_grid .+ α .* θ_init
v = ϕ_grid .- α .* θ_init

Δs = 1 / M
Δτ0 = 1e-2
N_iter = 500

lerp(a,b,t) = (1-t)*a + t*b

for n in 1:N_iter
    # ---------- pass 1: build λ(u^n,v^n), update u ----------
    p  = (u .+ v) ./ 2
    Pp = (u .- v) ./ (2α)                    # IMPORTANT

    gradθ = @. (-ℵ*p^3 + ℶ*p) + 2*ϵ*Pp
    gradϕ = @. (-3*ℵ*p^2 + ℶ) * Pp
    Hvals = @. ((-ℵ*p^3 + ℶ*p) * Pp) + ϵ*Pp^2
    fvals = @. α*gradϕ - gradθ
    fmid = (fvals[1:end-1] + fvals[2:end]) ./ 2

    dsϕ = diff(p) ./ Δs
    norm2 = @. dsϕ^2
    dot_term = @. gradθ[1:end-1] * dsϕ
    disc = @. dot_term^2 - 4*Hvals[1:end-1]*norm2
    λ_vals = @. (abs(dot_term) + sqrt(max(0.0, disc))) / (2*max(norm2, 1e-14))

    λmax = maximum(λ_vals)
    Δτ = min(Δτ0, 0.2 * Δs / max(λmax, 1e-12))   # CFL-like damping

    u_new = similar(u)
    u_new[end] = -v[end] + 2*ϕ_b
    @inbounds for i in M:-1:1
        u_new[i] = u[i] + (λ_vals[i] * (u_new[i+1]-u[i])/Δs + fmid[i]) * Δτ
    end

    # ---------- pass 2: build λ(u^{n+1},v^n), update v ----------
    p  = (u_new .+ v) ./ 2
    Pp = (u_new .- v) ./ (2α)

    gradθ = @. (-ℵ*p^3 + ℶ*p) + 2*ϵ*Pp
    gradϕ = @. (-3*ℵ*p^2 + ℶ) * Pp
    Hvals = @. ((-ℵ*p^3 + ℶ*p) * Pp) + ϵ*Pp^2
    gvals = @. α*gradϕ + gradθ
    gmid = (gvals[1:end-1] + gvals[2:end]) ./ 2

    dsϕ = diff(p) ./ Δs
    norm2 = @. dsϕ^2
    dot_term = @. gradθ[1:end-1] * dsϕ          # consistent indexing
    disc = @. dot_term^2 - 4*Hvals[1:end-1]*norm2
    λ_vals = @. (abs(dot_term) + sqrt(max(0.0, disc))) / (2*max(norm2, 1e-14))

    v_new = similar(v)
    v_new[1] = -u_new[1] + 2*ϕ_a
    @inbounds for i in 2:M+1
        λi = λ_vals[i-1]
        v_new[i] = v[i] - (λi * (v[i]-v_new[i-1])/Δs - gmid[i-1]) * Δτ
    end

    # ---------- robust 1D reparam on fixed φ-grid ----------
    ϕ = (u_new .+ v_new) ./ 2
    θ = (u_new .- v_new) ./ (2α)

    perm = sortperm(ϕ)
    ϕs = ϕ[perm]
    θs = θ[perm]

    θr = zeros(M+1)
    for j in 2:M
        φt = ϕ_grid[j]
        k = searchsortedfirst(ϕs, φt)
        if k <= 1
            θr[j] = θs[1]
        elseif k > M+1
            θr[j] = θs[end]
        else
            φ0, φ1 = ϕs[k-1], ϕs[k]
            τ = abs(φ1-φ0) < 1e-14 ? 0.0 : (φt-φ0)/(φ1-φ0)
            θr[j] = lerp(θs[k-1], θs[k], τ)
        end
    end
    θr[1] = θ[1]
    θr[end] = θ[end]

    u = ϕ_grid .+ α .* θr
    v = ϕ_grid .- α .* θr

    if n % 200 == 1
        pchk = (u .+ v) ./ 2
        Pchk = (u .- v) ./ (2α)
        Hchk = @. ((-ℵ*pchk^3 + ℶ*pchk) * Pchk) + ϵ*Pchk^2
        println("iter=$n, max|H|=", maximum(abs.(Hchk)))
    end
end

p = (u .+ v) ./ 2
P = (u .- v) ./ (2α);


In [ ]:
phi_all = -.3:0.001:.1
theta_all = -1:0.1:10
function H(p,P_p,params)
    ℵ, ℶ = params
    return P_p*(-ℵ*(p^2)*p + ℶ*p) +
           ϵ*P_p^2
end
H_val = [H(p, P_p, params) for p in phi_all, P_p in theta_all]


heatmap(theta_all,phi_all, H_val)
xlabel!(L"$P_p$")
ylabel!(L"$p$")
contour!(theta_all, phi_all, H_val; levels=[0.0], c=:white, lw=2)
plot!(zeros(length(p_stable)), p_stable, c="red", ls=:dash, label="Noise-free dissipation")
plot!(P,p, label="Solution of the GDA")
plot!(θ_best, ϕ_grid, label="Optimal path", ls=:dash)
xlims!(minimum(theta_all), maximum(theta_all))

In [ ]:
gradθ = @. (-ℵ*p^3 + ℶ*p) + 2*ϵ*P
gradϕ = @. (-3*ℵ*p^2 + ℶ) * P
Hvals = @. ((-ℵ*p^3 + ℶ*p) * P) + ϵ*P^2
fvals = @. α*gradϕ - gradθ

dsϕ = diff(p) ./ Δs
norm2 = @. dsϕ^2
dot_term = gradθ[1:end-1] .* dsϕ
disc = @. dot_term^2 - 4*Hvals[1:end-1]*norm2
λ_vals = @. (abs(dot_term) + sqrt(max(0.0, disc))) / (2*max(norm2, 1e-14))

λmax = maximum(λ_vals)
Δτ = min(Δτ0, 0.2 * Δs / max(λmax, 1e-12))   # CFL-like damping

fmid = (fvals[1:end-1] + fvals[2:end]) ./ 2
test = λ_vals .* (u[2:end] - u[1:end-1]) ./ Δs
plot(test + fmid)


In [ ]:
H_val = zeros(Float64,M+1)
for point in 1:M+1
    H_val[point] = H(p[point],P[point],params)
end
plot(H_val)
ylabel!("Hamiltonian")

In [ ]:
M = 300                    # start smaller, then increase
ϵ = 1e-3
ϕ_a = -sqrt(2*ε_d_p/β_b_p)
ϕ_b = 0.0
ϕ_grid = collect(range(ϕ_a, ϕ_b, length=M+1))

α = 0.01
θ_best = @. (ℵ*ϕ_grid^3 - ℶ*ϕ_grid) / ϵ   # = -b(ϕ)/ϵ
θ_init = zeros(M+1)
θ_init[1] = 0.0
θ_init[end] = 0.0

u = ϕ_grid .+ α .* θ_init
v = ϕ_grid .- α .* θ_init

Δs = 1 / M
Δτ0 = 1e-2
N_iter = 2000

lerp(a,b,t) = (1-t)*a + t*b

for n in 1:N_iter
    # ---------- pass 1: build λ(u^n,v^n), update u ----------
    p  = (u .+ v) ./ 2
    Pp = (u .- v) ./ (2α)                    # IMPORTANT

    gradθ = @. (-ℵ*p^3 + ℶ*p) + 2*ϵ*Pp
    gradϕ = @. (-3*ℵ*p^2 + ℶ) * Pp
    Hvals = @. ((-ℵ*p^3 + ℶ*p) * Pp) + ϵ*Pp^2
    fvals = @. α*gradϕ - gradθ
    fmid = (fvals[1:end-1] + fvals[2:end]) ./ 2

    dsϕ = diff(p) ./ Δs
    norm2 = @. dsϕ^2
    dot_term = @. gradθ[1:end-1] * dsϕ
    disc = @. dot_term^2 - 4*Hvals[1:end-1]*norm2
    λ_vals = @. (abs(dot_term) + sqrt(max(0.0, disc))) / (2*max(norm2, 1e-14))

    λmax = maximum(λ_vals)
    Δτ = min(Δτ0, 0.2 * Δs / max(λmax, 1e-12))   # CFL-like damping

    u_new = similar(u)
    u_new[end] = -v[end] + 2*ϕ_b
    @inbounds for i in M:-1:1
        u_new[i] = (u[i] / Δτ  + λ_vals[i] * u_new[i+1] /Δs + fmid[i]) / (1/Δτ + λ_vals[i]/Δs)
    end

    # ---------- pass 2: build λ(u^{n+1},v^n), update v ----------
    p  = (u_new .+ v) ./ 2
    Pp = (u_new .- v) ./ (2α)

    gradθ = @. (-ℵ*p^3 + ℶ*p) + 2*ϵ*Pp
    gradϕ = @. (-3*ℵ*p^2 + ℶ) * Pp
    Hvals = @. ((-ℵ*p^3 + ℶ*p) * Pp) + ϵ*Pp^2
    gvals = @. α*gradϕ + gradθ
    gmid = (gvals[1:end-1] + gvals[2:end]) ./ 2

    dsϕ = diff(p) ./ Δs
    norm2 = @. dsϕ^2
    dot_term = @. gradθ[1:end-1] * dsϕ          # consistent indexing
    disc = @. dot_term^2 - 4*Hvals[1:end-1]*norm2
    λ_vals = @. (abs(dot_term) + sqrt(max(0.0, disc))) / (2*max(norm2, 1e-14))

    v_new = similar(v)
    v_new[1] = -u_new[1] + 2*ϕ_a
    @inbounds for i in 2:M+1
        λi = λ_vals[i-1]
        v_new[i] = (v[i] / Δτ + λi * v_new[i-1]/Δs + gmid[i-1]) / (1/Δτ +λi / Δs)
    end

    # ---------- robust 1D reparam on fixed φ-grid ----------
    ϕ = (u_new .+ v_new) ./ 2
    θ = (u_new .- v_new) ./ (2α)

    perm = sortperm(ϕ)
    ϕs = ϕ[perm]
    θs = θ[perm]

    θr = zeros(M+1)
    for j in 2:M
        φt = ϕ_grid[j]
        k = searchsortedfirst(ϕs, φt)
        if k <= 1
            θr[j] = θs[1]
        elseif k > M+1
            θr[j] = θs[end]
        else
            φ0, φ1 = ϕs[k-1], ϕs[k]
            τ = abs(φ1-φ0) < 1e-14 ? 0.0 : (φt-φ0)/(φ1-φ0)
            θr[j] = lerp(θs[k-1], θs[k], τ)
        end
    end
    θr[1] = 0.0
    θr[end] = 0.0

    u = ϕ_grid .+ α .* θr
    v = ϕ_grid .- α .* θr

    if n % 200 == 1
        pchk = (u .+ v) ./ 2
        Pchk = (u .- v) ./ (2α)
        Hchk = @. ((-ℵ*pchk^3 + ℶ*pchk) * Pchk) + ϵ*Pchk^2
        println("iter=$n, max|H|=", maximum(abs.(Hchk)))
    end
end

p = (u .+ v) ./ 2
P = (u .- v) ./ (2α);


In [ ]:
phi_all = -.3:0.001:.1
theta_all = -1:0.1:10
function H(p,P_p,params)
    ℵ, ℶ = params
    return P_p*(-ℵ*(p^2)*p + ℶ*p) +
           ϵ*P_p^2
end
H_val = [H(p, P_p, params) for p in phi_all, P_p in theta_all]


heatmap(theta_all,phi_all, H_val)
xlabel!(L"$P_p$")
ylabel!(L"$p$")
contour!(theta_all, phi_all, H_val; levels=[0.0], c=:white, lw=2)
plot!(zeros(length(p_stable)), p_stable, c="red", ls=:dash, label="Noise-free dissipation")
plot!(P,p, label="Solution of the GDA")
plot!(θ_best, ϕ_grid, label="Optimal path", ls=:dash)
xlims!(minimum(theta_all), maximum(theta_all))

In [ ]:
H_val = zeros(Float64,M+1)
for point in 1:M+1
    H_val[point] = H(p[point],P[point],params)
end
plot(H_val)
ylabel!("Hamiltonian")

In [ ]:
# ---------------- basic physics functions ----------------
function gradθ(p, Pp, params)
    ϵ, ℵ, ℶ = params
    @. (-ℵ*p^3 + ℶ*p) + 2*ϵ*Pp
end

function gradϕ(p, Pp, params)
    ϵ, ℵ, ℶ = params
    @. (-3*ℵ*p^2 + ℶ) * Pp
end

function Hvals(p, Pp, params)
    ϵ, ℵ, ℶ = params
    @. ((-ℵ*p^3 + ℶ*p) * Pp) + ϵ*Pp^2
end

function build_lambda_and_mid(u, v, α, Δs, params; sign::Int)

    p  = (u .+ v) ./ 2
    Pp = (u .- v) ./ (2α)

    gθ = gradθ(p, Pp, params)
    gϕ = gradϕ(p, Pp, params)
    H  = Hvals(p, Pp, params)

    src = @. α*gϕ + sign*gθ
    srcmid = (src[1:end-1] .+ src[2:end]) ./ 2

    dsϕ = diff(p) ./ Δs
    norm2 = @. dsϕ^2
    dotterm = @. gθ[1:end-1] * dsϕ
    disc = @. dotterm^2 - 4*H[1:end-1]*norm2

    λ = @. (abs(dotterm) + sqrt(max(0.0, disc))) / (2*max(norm2, 1e-14))

    return λ, srcmid
end

function update_u(u, v, α, Δs, Δτ0, ϕ_b, params)
    λ, fmid = build_lambda_and_mid(u, v, α, Δs, params; sign = -1)

    λmax = maximum(λ)
    Δτ = min(Δτ0, 0.2 * Δs / max(λmax, 1e-12))

    u_new = similar(u)
    u_new[end] = -v[end] + 2*ϕ_b

    @inbounds for i in M:-1:1
        u_new[i] = (u[i]/Δτ + λ[i]*u_new[i+1]/Δs + fmid[i]) / (1/Δτ + λ[i]/Δs)
    end

    return u_new, Δτ
end

function update_v(u_new, v, α, Δs, Δτ, ϕ_a, params)
    λ, gmid = build_lambda_and_mid(u_new, v, α, Δs, params; sign = +1)

    v_new = similar(v)
    v_new[1] = -u_new[1] + 2*ϕ_a

    @inbounds for i in 2:M+1
        λi = λ[i-1]
        v_new[i] = (v[i]/Δτ + λi*v_new[i-1]/Δs + gmid[i-1]) /
                   (1/Δτ + λi/Δs)
    end

    return v_new
end



In [ ]:
@inline lerp(a,b,t) = (1-t)*a + t*b

"""
Reparameterize a discrete curve φ (d×N) to have uniform arc-length spacing.

Inputs
- φ::AbstractMatrix{T}: size (d, N), columns are points φ[:,i] ∈ R^d
- payload::AbstractArray: either
    * vector length N, or
    * matrix size (k, N) (any "data attached to points")
- Nout::Int: number of output points (default = N)

Options
- epslen: threshold to ignore zero-length segments
- endpoint_policy:
    * :keep -> keep endpoints exactly (default)
    * :wrap -> treat as periodic (not implemented here; easy to add)

Returns
- φr: (d, Nout) reparameterized points
- payloadr: payload interpolated to the new parameterization (same "shape", but Nout points)
"""
function reparametrization_arc_length(
    φ::AbstractMatrix{T},
    payload::AbstractArray,
    Nout::Int = size(φ, 2);
    epslen::T = T(1e-14),
    endpoint_policy::Symbol = :keep,
) where {T<:Real}

    d, N = size(φ)
    @assert N ≥ 2 "Need at least 2 points"
    @assert endpoint_policy == :keep "Only :keep implemented"

    # --- cumulative arc-length s[i] ---
    s = zeros(T, N)
    @inbounds for i in 2:N
        # Euclidean distance between consecutive points
        di = zero(T)
        @inbounds for k in 1:d
            δ = φ[k,i] - φ[k,i-1]
            di += δ*δ
        end
        di = sqrt(di)
        s[i] = s[i-1] + (di > epslen ? di : zero(T))
    end

    L = s[end]
    if L ≤ epslen
        # Degenerate curve: all points identical (or nearly). Return copies.
        φr = repeat(φ[:, 1:1], 1, Nout)
        payloadr = _payload_repeat_first(payload, Nout)
        return φr, payloadr
    end

    # Uniform target arc-length positions
    sr = range(zero(T), L, length = Nout)

    # Output containers
    φr = similar(φ, d, Nout)
    payloadr = _payload_alloc_like(payload, Nout)

    # --- helper: linear interpolation at target s ---
    # For each target sr[j], find segment i-1 -> i s.t. s[i-1] <= sr <= s[i]
    @inbounds for j in 1:Nout
        sj = sr[j]
        i = searchsortedfirst(s, sj)  # i in 1..N
        if i <= 1
            # clamp to first point
            @inbounds for k in 1:d
                φr[k,j] = φ[k,1]
            end
            _payload_set!(payloadr, payload, j, 1, one(T), zero(T))
        elseif i > N
            # clamp to last point
            @inbounds for k in 1:d
                φr[k,j] = φ[k,N]
            end
            _payload_set!(payloadr, payload, j, N, one(T), zero(T))
        else
            s0, s1 = s[i-1], s[i]
            t = (s1 - s0) ≤ epslen ? zero(T) : (sj - s0) / (s1 - s0)

            @inbounds for k in 1:d
                φr[k,j] = (one(T)-t)*φ[k,i-1] + t*φ[k,i]
            end
            _payload_set!(payloadr, payload, j, i, t, one(T)-t)
        end
    end

    return φr, payloadr
end

# ---- payload helpers (supports vector length N or matrix k×N) ----

# Allocate payload-like container with Nout points
function _payload_alloc_like(payload::AbstractVector{T}, Nout::Int) where {T}
    return similar(payload, T, Nout)
end
function _payload_alloc_like(payload::AbstractMatrix{T}, Nout::Int) where {T}
    k, _ = size(payload)
    return similar(payload, T, k, Nout)
end

# If curve is degenerate, repeat first payload entry
function _payload_repeat_first(payload::AbstractVector{T}, Nout::Int) where {T}
    out = similar(payload, T, Nout)
    out .= payload[1]
    return out
end
function _payload_repeat_first(payload::AbstractMatrix{T}, Nout::Int) where {T}
    k, _ = size(payload)
    out = similar(payload, T, k, Nout)
    @inbounds for j in 1:Nout
        out[:,j] .= payload[:,1]
    end
    return out
end

# Write interpolated payload at output index j.
# Convention: i is the RIGHT endpoint index (segment i-1 -> i).
function _payload_set!(out::AbstractVector{T}, payload::AbstractVector{T},
                      j::Int, i::Int, t::T, w0::T) where {T}
    # out[j] = (1-t)*payload[i-1] + t*payload[i]
    if i == 1
        out[j] = payload[1]
    else
        out[j] = w0*payload[i-1] + t*payload[i]
    end
end
function _payload_set!(out::AbstractMatrix{T}, payload::AbstractMatrix{T},
                      j::Int, i::Int, t::T, w0::T) where {T}
    k, _ = size(payload)
    if i == 1
        @inbounds for r in 1:k
            out[r,j] = payload[r,1]
        end
    else
        @inbounds for r in 1:k
            out[r,j] = w0*payload[r,i-1] + t*payload[r,i]
        end
    end
end

function reparametrization(u_new, v_new, α, M)
    # u_new, v_new: (N, d)
    N, d = size(u_new)
    @assert size(v_new) == (N, d)
    @assert N == M+1

    ϕ = (u_new .+ v_new) ./ 2          # (N, d)
    θ = (u_new .- v_new) ./ (2α)       # (N, d)

    # Convert to (d, N) for arc-length code
    φmat = permutedims(ϕ)              # (d, N)
    payload = permutedims(θ)           # (d, N)  (payload can be matrix k×N)

    # Reparametrize to N points (i.e. M+1)
    φr, θr = reparametrization_arc_length(φmat, payload, M+1)

    # Convert back to (N, d)
    ϕr = permutedims(φr)               # (N, d)
    θr = permutedims(θr)               # (N, d)

    u = ϕr .+ α .* θr
    v = ϕr .- α .* θr
    return u, v
end

In [ ]:
M = 300                    # start smaller, then increase
ϵ = 1e-3
params = (ϵ, ℵ, ℶ)
ϕ_a = -sqrt(2*ε_d_p/β_b_p)
ϕ_b = 0.0
ϕ_grid = collect(range(ϕ_a, ϕ_b, length=M+1))

α = 0.01
θ_best = @. (ℵ*ϕ_grid^3 - ℶ*ϕ_grid) / ϵ   # = -b(ϕ)/ϵ
θ_init = zeros(M+1)
θ_init[1] = 0.0
θ_init[end] = 0.0

u = ϕ_grid .+ α .* θ_init
v = ϕ_grid .- α .* θ_init

Δs = 1 / M
Δτ0 = 1e-2
N_iter = 2000

for n in 1:N_iter
    u_new, Δτ = update_u(u, v, α, Δs, Δτ0, ϕ_b, params)
    v_new     = update_v(u_new, v, α, Δs, Δτ, ϕ_a, params)
    u, v      = reparametrization(u_new, v_new, α, M)
end
p = (u .+ v) ./ 2
P = (u .- v) ./ (2α);

In [ ]:
phi_all = -.3:0.001:.1
theta_all = -1:0.1:10
function H(p,P_p,params)
    ϵ, ℵ, ℶ  = params
    return P_p*(-ℵ*(p^2)*p + ℶ*p) +
           ϵ*P_p^2
end
H_val = [H(p, P_p, params) for p in phi_all, P_p in theta_all]


heatmap(theta_all,phi_all, H_val)
xlabel!(L"$P_p$")
ylabel!(L"$p$")
contour!(theta_all, phi_all, H_val; levels=[0.0], c=:white, lw=2)
plot!(zeros(length(p_stable)), p_stable, c="red", ls=:dash, label="Noise-free dissipation")
plot!(P,p, label="Solution of the GDA")
plot!(θ_best, ϕ_grid, label="Optimal path", ls=:dash)
xlims!(minimum(theta_all), maximum(theta_all))

In [ ]:
# ---------------- basic physics functions ----------------
function grad_phi_H(ϕ::AbstractVector{<:Real}, θ::AbstractVector{<:Real}, params)
    x, p = ϕ
    P_x, P_p = θ
    ϵ,ℵ,ℶ = params
    grad =@. [-ℵ*(3x^2*P_x+p^2*P_x+2*x*p*P_p)-ℶ*P_x,
            -ℵ*(3p^2*P_p+x^2*P_p+2*p*x*P_x)+ℶ*P_p]
    return(grad)
end

function grad_theta_H(ϕ::AbstractVector{<:Real}, θ::AbstractVector{<:Real}, params)
    x, p = ϕ
    P_x, P_p = θ
    ϵ,ℵ,ℶ = params
    grad =@. [-ℵ*(x^2+p^2)x-ℶ*x+2*ϵ*P_x,
            -ℵ*(x^2+p^2)p+ℶ*p+2*ϵ*P_p,]
    return(grad)
end

function H_vals(ϕ::AbstractVector{<:Real}, θ::AbstractVector{<:Real}, params)
    x, p = ϕ
    P_x, P_p = θ
    ϵ,ℵ,ℶ = params
    return @. P_x*(-ℵ*(x^2+p^2)*x - ℶ*x) +
           P_p*(-ℵ*(x^2+p^2)*p + ℶ*p) +
           ϵ*(P_x^2 + P_p^2)
end

function build_lambda_and_mid(u, v, α, Δs, params; sign::Int)
    # u, v are (N, d) with N = M+1
    N, d = size(u)
    @assert size(v) == (N, d)

    # ϕ, θ are (N, d)
    ϕ = (u .+ v) ./ 2
    θ = (u .- v) ./ (2α)

    # gradients along nodes: (N, d)
    gθ = similar(u)
    gϕ = similar(u)

    # Hamiltonian along nodes: (N,)
    H = zeros(eltype(u), N)

    @inbounds for i in 1:N
        ϕi = collect(@view ϕ[i, :])   # vector length d (safe w/ your current signatures)
        θi = collect(@view θ[i, :])

        gθ[i, :] .= grad_theta_H(ϕi, θi, params)
        gϕ[i, :] .= grad_phi_H(ϕi, θi, params)
        H[i] = H_vals(ϕi, θi, params)
    end

    # src at nodes: (N, d), midpoint source on segments: (N-1, d)
    src    = α .* gϕ .+ sign .* gθ
    srcmid = (src[1:end-1, :] .+ src[2:end, :]) ./ 2

    # segment differences in ϕ-space: dϕ[i,:] = (ϕ[i+1]-ϕ[i]) / Δs, shape (N-1, d)
    dϕ = (ϕ[2:end, :] .- ϕ[1:end-1, :]) ./ Δs

    # norm2[i] = ||dϕ[i]||^2, dotterm[i] = <gθ[i], dϕ[i]>
    norm2   = zeros(eltype(u), N-1)
    dotterm = zeros(eltype(u), N-1)

    @inbounds for i in 1:N-1
        s_norm2 = zero(eltype(u))
        s_dot   = zero(eltype(u))
        @inbounds for k in 1:d
            δ = dϕ[i, k]
            s_norm2 += δ*δ
            s_dot   += gθ[i, k]*δ
        end
        norm2[i]   = s_norm2
        dotterm[i] = s_dot
    end

    disc = @. dotterm^2 - 4 * H[1:end-1] * norm2

    # λ on segments: (N-1,)
    λ = similar(norm2)
    @inbounds for i in 1:N-1
        denom = 2 * max(norm2[i], 1e-14)
        λ[i] = (abs(dotterm[i]) + sqrt(max(0.0, disc[i]))) / denom
    end

    return λ, srcmid
end


function update_u(u, v, α, Δs, Δτ0, ϕ_b, params)
    # u, v are (N,2). ϕ_b should be length-2 (or 1×2) endpoint in ϕ-space.
    N = size(u, 1)
    M = N - 1

    λ, fmid = build_lambda_and_mid(u, v, α, Δs, params; sign = -1)

    λmax = maximum(λ)
    Δτ = min(Δτ0, 0.2 * Δs / max(λmax, 1e-12))

    u_new = similar(u)

    # boundary: u_new[end,:] = -v[end,:] + 2*ϕ_b
    u_new[end, :] .= -v[end, :] .+ 2 .* ϕ_b

    @inbounds for i in M:-1:1
        denom = (1/Δτ + λ[i]/Δs)
        u_new[i, :] .= (u[i, :]./Δτ .+ (λ[i]/Δs) .* u_new[i+1, :] .+ fmid[i, :]) ./ denom
    end

    return u_new, Δτ
end


function update_v(u_new, v, α, Δs, Δτ, ϕ_a, params)
    # u_new, v are (N,2). ϕ_a should be length-2 (or 1×2) endpoint in ϕ-space.
    N = size(v, 1)
    M = N - 1

    λ, gmid = build_lambda_and_mid(u_new, v, α, Δs, params; sign = +1)

    v_new = similar(v)

    # boundary: v_new[1,:] = -u_new[1,:] + 2*ϕ_a
    v_new[1, :] .= -u_new[1, :] .+ 2 .* ϕ_a

    @inbounds for i in 2:N
        λi = λ[i-1]
        denom = (1/Δτ + λi/Δs)
        v_new[i, :] .= (v[i, :]./Δτ .+ (λi/Δs) .* v_new[i-1, :] .+ gmid[i-1, :]) ./ denom
    end

    return v_new
end

In [ ]:
M = 300                    # start smaller, then increase
ϵ = 1e-3
params = (ϵ, ℵ, ℶ)
ϕ_a = (0.0, -sqrt(2*ε_d_p/β_b_p))
ϕ_b = (0.0, 0.0)
ϕ_grid = collect(range(ϕ_a[2], ϕ_b[2], length=M+1))


# Your parabola: x as function of p (endpoints give x=0 automatically)
#initial_path_relation(p; c=2.0) = c * p * (p - ϕ_a[2])  # x(p)
function initial_path_relation(p; A=0.3, h=0.15)
    p_a = ϕ_a[2]
    t = (p - p_a)/(-p_a)  # 0..1
    return (-p_a) * (A*sin(pi*t) + h*sin(2pi*t))
end

function initial_guess_path(M, ϕ_a, ϕ_b; n_fine=50_000)
    p_a = ϕ_a[2]
    p_b = ϕ_b[2]  # should be 0.0 here

    # 1) Fine sampling in parameter p
    p_f = collect(range(p_a, p_b, length=n_fine))
    x_f = similar(p_f)
    @inbounds for k in eachindex(p_f)
        x_f[k] = initial_path_relation(p_f[k])   # <-- use your new relation here
    end

    # 2) Cumulative arc-length along the polyline
    s = zeros(Float64, n_fine)
    @inbounds for k in 2:n_fine
        dx = x_f[k] - x_f[k-1]
        dp = p_f[k] - p_f[k-1]
        s[k] = s[k-1] + sqrt(dx*dx + dp*dp)
    end
    L = s[end]

    # 3) Target equally spaced arc-lengths
    s_target = collect(range(0.0, L, length=M+1))

    # 4) Interpolate (x,p) at those arc-lengths
    x = Vector{Float64}(undef, M+1)
    p = Vector{Float64}(undef, M+1)

    @inbounds for i in 1:(M+1)
        st = s_target[i]
        k = searchsortedfirst(s, st)

        if k <= 1
            x[i] = x_f[1];  p[i] = p_f[1]
        elseif k > n_fine
            x[i] = x_f[end]; p[i] = p_f[end]
        else
            s0, s1 = s[k-1], s[k]
            t = (st - s0) / (s1 - s0)  # in [0,1]
            x[i] = (1-t)*x_f[k-1] + t*x_f[k]
            p[i] = (1-t)*p_f[k-1] + t*p_f[k]
        end
    end

    # Pack as a 2×(M+1) matrix: row1=x, row2=p
    path = permutedims(hcat(x, p))  # (M+1)×2
    path = permutedims(path)        # 2×(M+1)

    # Enforce exact endpoints (good practice)
    path[1, :] .= ϕ_a
    path[end, :] .= ϕ_b

    return path
end

ϕ_init = initial_guess_path(M, ϕ_a, ϕ_b)

# Quick check: step lengths should be (nearly) constant
gaps = [norm(ϕ_init[i+1,:] - ϕ_init[i,:]) for i in 1:M]
println("gap min/max = ", minimum(gaps), " / ", maximum(gaps))
println("relative spread = ", (maximum(gaps)-minimum(gaps))/(sum(gaps)/length(gaps)))

In [ ]:
α = 0.01
θ_best = @. (ℵ*ϕ_grid^3 - ℶ*ϕ_grid) / ϵ   # = -b(ϕ)/ϵ
θ_init = zeros(M+1,length(ϕ_a))

u = ϕ_init .+ α .* θ_init
v = ϕ_init .- α .* θ_init

Δs = 1 / M
Δτ0 = 1e-2
N_iter = 10000

for n in 1:N_iter
    u_new, Δτ = update_u(u, v, α, Δs, Δτ0, ϕ_b, params)
    v_new     = update_v(u_new, v, α, Δs, Δτ, ϕ_a, params)
    u, v      = reparametrization(u_new, v_new, α, M)
end
ϕ = (u .+ v) ./ 2
θ = (u .- v) ./ (2α);
p = ϕ[:,2]
P = θ[:,2];

In [ ]:
phi_all = -.3:0.001:.1
theta_all = -1:0.1:10
function H(p,P_p,params)
    ϵ, ℵ, ℶ  = params
    return P_p*(-ℵ*(p^2)*p + ℶ*p) +
           ϵ*P_p^2
end
H_val = [H(p, P_p, params) for p in phi_all, P_p in theta_all]


heatmap(theta_all,phi_all, H_val)
xlabel!(L"$P_p$")
ylabel!(L"$p$")
contour!(theta_all, phi_all, H_val; levels=[0.0], c=:white, lw=2)
plot!(zeros(length(p_stable)), p_stable, c="red", ls=:dash, label="Noise-free dissipation")
plot!(P,p, label="Solution of the GDA")
plot!(θ_best, ϕ_grid, label="Optimal path", ls=:dash)
xlims!(minimum(theta_all), maximum(theta_all))

In [ ]:
function grad_phi_H(x_a,p_a,x_b,p_b,P_x_a,P_p_a,P_x_b,P_p_b,params)
    β_a_p,β_b_p,ε_d_p,κ_b_p,_ = params
    grad = [β_a_p*(p_b*P_x_a - x_b*P_p_a)/sqrt(2)+β_b_p*(p_a*P_x_b-x_a*P_p_b)/sqrt(2),
            -β_a_p*(x_b*P_x_a + p_b*P_p_a)/sqrt(2)+β_b_p*(x_a*P_x_b+p_a*P_p_b)/sqrt(2),
            -β_a_p*(p_a*P_x_a+x_a*P_p_a)/sqrt(2)-κ_b_p*P_x_b,
            β_a_p*(x_a*P_x_a-p_a*P_p_a)/sqrt(2)-κ_b_p*P_p_b]
    return(grad)
end

function grad_theta_H(x_a,p_a,x_b,p_b,P_x_a,P_p_a,P_x_b,P_p_b,params)
    β_a_p,β_b_p,ε_d_p,κ_b_p,_ = params
    grad = [β_a_p*(x_a*p_b-p_a*x_b)/sqrt(2),
            -β_a_p*(x_a*x_b+p_a*p_b)/sqrt(2),
            β_b_p*x_a*p_a/sqrt(2)-κ_b_p*x_b+2*ϵ*P_x_b,
            -β_b_p*(x_a^2-p_a^2)/2/sqrt(2)-ε_d_p/sqrt(2)-κ_b_p*p_b+2*ϵ*P_p_b
            ]
    return(grad)
end

function H(x_a,p_a,x_b,p_b,P_x_a,P_p_a,P_x_b,P_p_b,params)
    β_a_p,β_b_p,ε_d_p,κ_b_p,ϵ = params
    
    return (β_a_p*(x_a*p_b-p_a*x_b)/sqrt(2))*P_x_a +
           (-β_a_p*(x_a*x_b+p_a*p_b)/sqrt(2))*P_p_a +
            (β_b_p*x_a*p_a/sqrt(2)-κ_b_p*x_b)*P_x_b +
            (-β_b_p*(x_a^2-p_a^2)/2/sqrt(2)-ε_d_p/sqrt(2)-κ_b_p*p_b)*P_p_b +
            ϵ*(P_x_b^2+P_p_b^2)
end

In [ ]:
# ---------------- basic physics functions ----------------
function grad_phi_H(ϕ::AbstractVector{<:Real}, θ::AbstractVector{<:Real}, params)
    x_a, p_a, x_a, p_b = ϕ
    P_x_a, P_p_a, P_x_b, P_p_b = θ
    ϵ,ℵ,ℶ = params
    grad =@. [-ℵ*(3x^2*P_x+p^2*P_x+2*x*p*P_p)-ℶ*P_x,
            -ℵ*(3p^2*P_p+x^2*P_p+2*p*x*P_x)+ℶ*P_p]
    return(grad)
end

function grad_theta_H(ϕ::AbstractVector{<:Real}, θ::AbstractVector{<:Real}, params)
    x, p = ϕ
    P_x, P_p = θ
    ϵ,ℵ,ℶ = params
    grad =@. [-ℵ*(x^2+p^2)x-ℶ*x+2*ϵ*P_x,
            -ℵ*(x^2+p^2)p+ℶ*p+2*ϵ*P_p,]
    return(grad)
end

function H_vals(ϕ::AbstractVector{<:Real}, θ::AbstractVector{<:Real}, params)
    x, p = ϕ
    P_x, P_p = θ
    ϵ,ℵ,ℶ = params
    return @. P_x*(-ℵ*(x^2+p^2)*x - ℶ*x) +
           P_p*(-ℵ*(x^2+p^2)*p + ℶ*p) +
           ϵ*(P_x^2 + P_p^2)
end

In [ ]:
M = 500
ϵ = 1e-3
ϕ_a = [0.0, -sqrt(2*ε_d_p/β_b_p), 0.0 ,0.0]
ϕ_b = [0.0, 0.0, 0.0, -sqrt(0.5)*ε_d_p/κ_b_p]
params = (ϵ,β_a_p,β_b_p,ε_d_p,κ_b_p)
ϕ_grid = collect(range(ϕ_a[2], ϕ_b[2], length=M+1))

ϕ_init = initial_guess_path(M, ϕ_a, ϕ_b)

# Quick check: step lengths should be (nearly) constant
gaps = [norm(ϕ_init[i+1,:] - ϕ_init[i,:]) for i in 1:M]
println("gap min/max = ", minimum(gaps), " / ", maximum(gaps))
println("relative spread = ", (maximum(gaps)-minimum(gaps))/(sum(gaps)/length(gaps)))